# Exercise 3 · Make Codex's browser test find elements as a user would

You will have Codex rewrite its own browser test to find the page as a user does.

**Goal:** Have Codex rewrite a browser test so it finds elements as a user would and waits with `expect`.

**Why it matters:** A browser (end-to-end) test checks the page itself, which a unit test cannot.

**Before you start:**
- You installed Codex and signed in (course README, step 1).
- Your computer can download about 570 MB. Step 3 downloads a browser for tests, once.

**What the check measures:** `test_page.py` finds by label and test id, and waits with `expect`. It has no fixed wait and no CSS path. Your last run passed. Your Step 7 answer is right.

**Done when:** the last cell prints `Done:`.

## The situation

`cases.html` is a case list with a Status filter. `test_page.py` is Codex's first draft of its test. The files hold made-up data.

## Step 1 · Write the practice files

Run the cell. It writes `cases.html`, `test_page.py` and `.gitignore` into `practice/`, beside this notebook. It never overwrites a file.

Expect three lines that start with `Wrote practice/`, or `Kept` for a file that is already there.

In [ ]:
FILES = {
    "cases.html": """<!doctype html>
<html><head><meta charset="utf-8"><title>Case list</title></head>
<body>
<label for="status">Status</label>
<select id="status"><option value="">All</option><option value="open">Open</option><option value="closed">Closed</option></select>
<table><tbody id="cases"></tbody></table>
<script>
const CASES = [{id: "C-1", status: "open"}, {id: "C-2", status: "open"}, {id: "C-3", status: "closed"}];
function show(status) {
  const body = document.getElementById("cases");
  clearTimeout(window.pending);
  body.innerHTML = "";
  window.pending = setTimeout(() => {
    for (const c of CASES.filter(c => !status || c.status === status)) {
      const row = document.createElement("tr");
      row.dataset.testid = "case-row";
      row.innerHTML = `<td>${c.id}</td><td>${c.status === "open" ? "Open" : "Closed"}</td>`;
      body.appendChild(row);
    }
  }, 300);
}
document.getElementById("status").addEventListener("change", e => show(e.target.value));
show("");
</script></body></html>
""",
    "test_page.py": """# Codex's first draft of a test for the Status filter (simulated).
from pathlib import Path

from playwright.sync_api import Page

PAGE = Path(__file__).parent / "cases.html"


def test_filter_shows_only_open_cases(page: Page) -> None:
    page.goto(PAGE.as_uri())
    page.locator("#status").select_option("open")
    page.wait_for_timeout(2000)
    assert "Open" in page.locator("#cases > tr:nth-child(2)").inner_text()
    assert page.locator("#cases > tr:nth-child(3)").count() == 0
""",
    ".gitignore": "__pycache__/\n.pytest_cache/\nresults.xml\ntest-results/\n",
}

from pathlib import Path

PRACTICE = Path("practice")
PRACTICE.mkdir(exist_ok=True)
for name, text in FILES.items():
    if (PRACTICE / name).exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        (PRACTICE / name).write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

## Step 2 · Open a terminal and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-03/exercise-03-find-elements-as-a-user/practice
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect no output from the last line. The commit lets you undo any change.

## Step 3 · Install the browser for tests (once)

`uv run --with pytest-playwright==0.10.0` adds Playwright for one command only. It changes no course file.

```sh
uv run --with pytest-playwright==0.10.0 playwright install chromium
```

Expect download lines for Chrome for Testing, Chrome Headless Shell and FFmpeg, about 570 MB in all. If it stops with an error, ask your trainer.

## Step 4 · Run Codex's draft

```sh
uv run --with pytest-playwright==0.10.0 pytest test_page.py
```

Expect `1 passed` after about 2.5 seconds. No window opens: the test runs headless (exercise 4 explains).

Read `test_page.py`. It finds the rows with a CSS path, `#cases > tr:nth-child(3)`: a selector that names an element by its place in the page's HTML. It waits a fixed two seconds.

### Learn: a passing draft can still be the wrong test

A CSS path breaks when the HTML changes. Playwright's guide: "CSS and XPath are not recommended as the DOM can often change leading to non resilient tests" ([Locators](https://playwright.dev/python/docs/locators)). A fixed wait is slow and still fails on a slower page: "Never wait for timeout in production. Tests that wait for time are inherently flaky" ([Page](https://playwright.dev/python/docs/api/class-page#page-wait-for-timeout)).

Better: find the control by its label, as a user does, and the rows by a test id. A test id is an attribute the page sets for tests. In `cases.html`, `row.dataset.testid = "case-row"` gives each row `data-testid="case-row"`.

## Step 5 · Ask Codex to rewrite the test

First write the expected counts yourself. Open `cases.html` and read the `CASES` list. After choosing Open, how many rows should show? How many of them should say Closed? Put your two numbers in place of `__` below.

```sh
codex --no-daemon
```

If Codex stops with an error about `--no-daemon`, run `codex` without it. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Then type the request with your numbers and press Enter:

```text
Rewrite test_page.py. Find the Status control by its label and the case rows by the test id case-row. After choosing Open, expect __ rows, and __ rows with the text Closed. Use expect(...).to_have_count(...), which waits. Remove the fixed wait and every CSS path. Change only test_page.py. Do not run it. I will run it myself.
```

The counts come from the data, not from Codex. Expect Codex to edit `test_page.py`.

<details><summary>Show the answer</summary>

2 rows and 0 rows. Why: `CASES` holds C-1 and C-2 with status open, and C-3 with status closed.

</details>

## Step 6 · Read the rewrite, run it, commit it

Type `/diff`, read the change, press `q`, then type `/exit`. Run the test. `--junitxml=results.xml` saves the result in a file, for the check cell:

```sh
uv run --with pytest-playwright==0.10.0 pytest test_page.py --junitxml=results.xml
```

Expect `1 passed`. Keep the rewrite only if it passes and has no `wait_for_timeout`, `#cases` or `nth-child`. Otherwise run `git restore test_page.py` and go back to Step 5. Then commit:

```sh
git commit -qam "Find the filter's elements as a user would"
```

<details><summary>Show the answer</summary>

Keep a rewrite with `get_by_label("Status")`, `get_by_test_id("case-row")` and `expect(rows).to_have_count(2)`. Why: `expect` checks again until the count is right or its timeout ends ([Assertions](https://playwright.dev/python/docs/test-assertions)).

</details>

## Step 7 · Break the page on purpose, then answer

Open `practice/cases.html`. Change `<label for="status">Status</label>` to `<label for="status">State</label>`. Save it. Run the test again, with no result file:

```sh
uv run --with pytest-playwright==0.10.0 pytest test_page.py
```

Expect `1 failed` after about 30 seconds, with `TimeoutError` and `waiting for get_by_label("Status")`. Codex's draft passed on this page: it used the id `#status`. Undo the change:

```sh
git restore cases.html
```

What did your browser test just catch?

- **A.** The page lost the label a user reads to find the control.
- **B.** The filter returned the wrong cases.
- **C.** Nothing: the test failed by chance.

Write your letter between the quotes, then run the cell. Expect `Saved your answer: ` and your letter.

<details><summary>Show the answer</summary>

A. Why: the cases did not change, only the label did. A unit test of the filter function would still pass: it never opens the page.

</details>

In [ ]:
ANSWER = ""
print(f"Saved your answer: {ANSWER}")

## Check your work

Run the cell. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do. The check starts no browser.

In [ ]:
# Check your work: read test_page.py and results.xml, and compare your answer
import xml.etree.ElementTree as ET
from pathlib import Path

PRACTICE = Path("practice")
BANNED = ["wait_for_timeout", "sleep(", "nth-child", "#cases"]
WANTED = ['get_by_label("Status")', 'get_by_test_id("case-row")', "expect(", ".to_have_count("]


def first_problem() -> str:
    test, results = PRACTICE / "test_page.py", PRACTICE / "results.xml"
    if not test.is_file():
        return "practice/test_page.py is missing. Run the Step 1 cell."
    text = test.read_text(encoding="utf-8").replace("'", '"')
    for word in BANNED + WANTED:
        if (word in text) == (word in BANNED):
            return f"test_page.py {'still has' if word in BANNED else 'has no'} {word}. Ask Codex (Step 5)."
    print("ok: test_page.py finds by label and test id, waits with expect, and has no fixed wait")
    if not results.is_file() or results.stat().st_mtime <= test.stat().st_mtime:
        return "no results.xml newer than test_page.py. Run the test with --junitxml=results.xml (Step 6)."
    root = ET.parse(results).getroot()
    suites = [root] if root.tag == "testsuite" else root.findall("testsuite")
    got = {key: sum(int(s.get(key, 0)) for s in suites) for key in ("tests", "failures", "errors", "skipped")}
    if got["failures"] or got["errors"] or got["tests"] - got["skipped"] < 1:
        return (f"your last run had {got['failures']} failed and {got['errors']} errors, or no test passed. "
                "Read its output, then go back to Step 5.")
    print(f"ok: results.xml shows {got['tests']} passed, 0 failed")
    choice = str(globals().get("ANSWER", "")).strip().upper()
    if choice != "A":
        return f"answer {choice or 'none'} is not right. Write a letter in the Step 7 cell, run it, then this cell."
    print("ok: your answer is right")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run the test again (Step 6)."
print(f"Not yet: {problem}" if problem else
      "Done: your browser test finds the page as a user does, and passes.")

## What you learned

- A browser test reaches the page a user sees. Yours failed when the label changed.
- Keep a generated browser test only if it finds by label or test id and waits with `expect`.

**Reflect:** Which page in your own work has no test through the browser? What would a user do on it first?